<div style="background:#111A33;border-radius:14px;padding:26px 30px;border-left:6px solid #E5484D">
<div style="color:#F5A623;font-size:13px;letter-spacing:3px;font-weight:700">HOMEWORK &nbsp;&middot;&nbsp; WEEK 5 &nbsp;&middot;&nbsp; SQL FUNDAMENTALS</div>
<div style="color:#FFFFFF;font-size:34px;font-weight:800;line-height:1.15;margin-top:8px">Week 5 Homework</div>
<div style="color:#C7CEDB;font-size:16px;margin-top:6px">Sessions 9 &amp; 10 &middot; 100 marks</div>
<div style="background:#F5A623;color:#111A33;display:inline-block;padding:5px 14px;border-radius:6px;font-weight:700;font-size:12px;letter-spacing:1px;margin-top:16px">MODERN DATA ENGINEERING &nbsp;|&nbsp; DEPTHWARE</div>
</div>

**Name:** *(Mahfuzatul Bushra*)

---


## Setup &mdash; run this cell first

Loads the four tables and defines `q()`. Make sure the `data/` folder sits next to this notebook.

In [2]:
import duckdb
import pandas as pd
pd.set_option("display.max_rows", 20)

con = duckdb.connect()
for table in ["cities", "drivers", "riders", "rides"]:
    con.sql(f"DROP TABLE IF EXISTS {table}")
    con.sql(f"""
        CREATE TABLE {table} AS
        SELECT * FROM read_csv('data/{table}.csv', header = true)
    """)

def q(sql):
    """Run a SQL string and return the result as a pandas DataFrame."""
    return con.sql(sql).df()

print("Tables loaded:", [r[0] for r in con.sql("SHOW TABLES").fetchall()])

Tables loaded: ['cities', 'drivers', 'riders', 'rides']


---
## Section A &mdash; Session 9 foundations (30 marks)

### Q1 (5 marks) &mdash; `SELECT` + `WHERE` + `ORDER BY`

List the `ride_id`, `distance_km`, and `fare` of all **completed** rides between **25 and 30 km** (inclusive). Order by fare descending. Show the top 5.

In [3]:
q("""
    SELECT ride_id, distance_km, fare
    FROM rides
    WHERE status = 'completed' 
        AND distance_km BETWEEN 25 AND 30
    ORDER BY fare DESC
    LIMIT 5
""")


,ride_id,distance_km,fare
0,845,29.9,549.34
1,700,29.1,528.64
2,36,29.4,526.37
3,244,28.0,511.39
4,904,27.9,506.89


### Q2 (5 marks) &mdash; `IS NULL`

**(a)** Count how many rides have a missing `driver_id`.
**(b)** List **any 3** such rides showing `ride_id`, `rider_id`, and `status`.

In [4]:
# Q2(a) 
q("""
    SELECT COUNT(*) AS rides_with_missing_driver
    FROM rides
    WHERE driver_id IS NULL
""")


,rides_with_missing_driver
0,124


In [5]:
# Q2(b) 
q("""
    SELECT ride_id, rider_id, status
    FROM rides
    WHERE driver_id IS NULL
    LIMIT 3
""")

,ride_id,rider_id,status
0,29,35,cancelled
1,32,64,cancelled
2,35,35,cancelled


### Q3 (10 marks) &mdash; Aggregates with `NULL` handling

In **one query**, compute over the **entire rides table**: `total_rides`, `rides_with_fare`, `revenue` (rounded 0), `avg_fare` (rounded 2), `max_fare`, `min_fare`. Then explain briefly why `total_rides` and `rides_with_fare` differ.

In [6]:
# Q3 
q("""
    SELECT 
        COUNT(*) AS total_rides,
        COUNT(fare) AS rides_with_fare,
        ROUND(SUM(fare)) AS revenue,
        ROUND(AVG(fare), 2) AS avg_fare,
        MAX(fare) AS max_fare,
        MIN(fare) AS min_fare
    FROM rides
""")


,total_rides,rides_with_fare,revenue,avg_fare,max_fare,min_fare
0,1200,1076,240522.0,223.53,577.68,28.08


**Q3 explanation:**

This query provides a summary of ride fares, including the total number of rides, total revenue, average fare, and the highest and lowest fares. Out of **1,200 rides**, **1,076** have fare values, meaning **124 rides have missing (NULL) fare records.*

### Q4 (5 marks) &mdash; `GROUP BY` + `HAVING`

For **completed rides only**, group by `ride_date` and show date, ride count, and rounded revenue. Show only days with **more than 17 rides**, ordered by ride count descending.

In [7]:
# Q4
q("""
    SELECT
        ride_date,
        COUNT(*) AS ride_count,
        ROUND(SUM(fare)) AS revenue
    FROM rides
    WHERE status = 'completed'
    GROUP BY ride_date
    HAVING COUNT(*) > 17
    ORDER BY ride_count DESC
""")


,ride_date,ride_count,revenue
0,2025-02-08,20,4753.0
1,2025-02-28,20,4292.0
2,2025-03-02,19,3748.0
3,2025-03-23,19,3883.0
4,2025-01-26,18,4170.0


### Q5 (5 marks) &mdash; Conceptual: `WHERE` vs `HAVING`

The query below fails. Explain why in 1-2 sentences, then provide a corrected version that returns the same intent (drivers with more than 20 completed rides).

```sql
SELECT driver_id, COUNT(*) AS rides
FROM rides
WHERE status = 'completed'
  AND COUNT(*) > 20
GROUP BY driver_id
```

**Q5 explanation:**

*COUNT(*) is an aggregate function, so it cannot be used in the WHERE clause because WHERE filters rows before grouping and aggregation. To filter groups based on an aggregate value, use the HAVING clause after GROUP BY.*

In [8]:
# Q5 
q("""
    SELECT
        driver_id,
        COUNT(*) AS rides
    FROM rides
    WHERE status = 'completed'
    GROUP BY driver_id
    HAVING COUNT(*) > 20
""")


,driver_id,rides
0,6,22
1,7,22
2,8,21
3,10,22
4,14,23
...,...,...
23,44,23
24,45,29
25,46,23
26,48,22


---
## Section B &mdash; JOINs & CTEs (35 marks)

### Q6 (10 marks) &mdash; `INNER JOIN` + `GROUP BY`

Top 10 drivers by revenue from completed rides. Show `driver_id`, name, vehicle type, ride count, rounded revenue. Order by revenue descending.

> **Hint:** Include `driver_id` in your `GROUP BY` &mdash; two different drivers can share a name in this dataset.

In [9]:
# Q6 
q("""
    SELECT
        r.driver_id,
        d.name,
        d.vehicle_type,
        COUNT(*) AS ride_count,
        ROUND(SUM(r.fare)) AS revenue
    FROM rides AS r
    INNER JOIN drivers AS d
        ON r.driver_id = d.driver_id
    WHERE r.status = 'completed'
    GROUP BY
        r.driver_id,
        d.name,
        d.vehicle_type
    ORDER BY revenue DESC
    LIMIT 10
""")


,driver_id,name,vehicle_type,ride_count,revenue
0,29,Gizem Yilmaz,xl,33,10845.0
1,14,Buse Aslan,xl,23,8494.0
2,21,Emre Sahin,comfort,31,7495.0
3,37,Selim Yilmaz,comfort,24,7175.0
4,33,Kaan Ozdemir,comfort,27,6521.0
5,50,Buse Arslan,comfort,29,6487.0
6,7,Zeynep Polat,comfort,22,6251.0
7,15,Yagmur Aydin,comfort,23,6167.0
8,32,Melis Celik,comfort,24,6155.0
9,47,Melis Yilmaz,xl,18,6051.0


### Q7 (10 marks) &mdash; `LEFT JOIN`

For each of the 5 cities: number of riders based there (`riders`) and number of completed rides those riders took (`completed_rides`). Use a `LEFT JOIN` so all 5 cities appear. Order by `completed_rides` descending.

In [10]:
# Q7
q("""
    SELECT
        c.city_name,
        COUNT(DISTINCT rd.rider_id) AS riders,
        COUNT(r.ride_id) AS completed_rides
    FROM cities AS c
    LEFT JOIN riders AS rd
        ON c.city_id = rd.city_id
    LEFT JOIN rides AS r
        ON rd.rider_id = r.rider_id
       AND r.status = 'completed'
    GROUP BY
        c.city_id,
        c.city_name
    ORDER BY completed_rides DESC
""")


,city_name,riders,completed_rides
0,Izmir,20,227
1,Istanbul,19,216
2,Ankara,20,210
3,Antalya,21,196
4,Bursa,18,191


### Q8 (15 marks) &mdash; CTE chain

Using a CTE:
1. Build a per-driver summary of completed rides only: `rides`, `revenue`, `avg_fare`.
2. From that summary, return only drivers whose **average fare exceeds 250 TRY**.

Show `driver_id`, `rides`, `revenue` (rounded 0), `avg_fare` (rounded 2). Order by `avg_fare` descending.

In [11]:
# Q8 
q("""
    WITH driver_summary AS (
        SELECT
            driver_id,
            COUNT(*) AS rides,
            SUM(fare) AS revenue,
            AVG(fare) AS avg_fare
        FROM rides
        WHERE status = 'completed'
        GROUP BY driver_id
    )
    SELECT
        driver_id,
        rides,
        ROUND(revenue) AS revenue,
        ROUND(avg_fare, 2) AS avg_fare
    FROM driver_summary
    WHERE avg_fare > 250
    ORDER BY avg_fare DESC
""")


,driver_id,rides,revenue,avg_fare
0,14,23,8494.0,369.30
1,47,18,6051.0,336.16
2,29,33,10845.0,328.65
3,34,17,5300.0,311.75
4,37,24,7175.0,298.95
5,28,20,5936.0,296.82
6,42,17,5023.0,295.46
7,11,16,4696.0,293.52
8,7,22,6251.0,284.16
9,31,19,5102.0,268.51


---
## Section C &mdash; Window functions (25 marks)

### Q9 (10 marks) &mdash; Ranking comparison

CTE: total revenue per driver (completed rides only). Then add three ranking columns ordered by revenue descending: `row_number`, `rank_`, `dense_rank_`. Show the top 10. Then explain when the three rankings would produce **different** values.

In [12]:
# Q9
q("""
    WITH driver_revenue AS (
        SELECT
            driver_id,
            ROUND(SUM(fare)) AS revenue
        FROM rides
        WHERE status = 'completed'
        GROUP BY driver_id
    )
    SELECT
        driver_id,
        revenue,
        ROW_NUMBER() OVER (ORDER BY revenue DESC) AS row_number,
        RANK() OVER (ORDER BY revenue DESC) AS rank_,
        DENSE_RANK() OVER (ORDER BY revenue DESC) AS dense_rank_
    FROM driver_revenue
    ORDER BY revenue DESC
    LIMIT 10
""")

,driver_id,revenue,row_number,rank_,dense_rank_
0,29,10845.0,1,1,1
1,14,8494.0,2,2,2
2,21,7495.0,3,3,3
3,37,7175.0,4,4,4
4,33,6521.0,5,5,5
5,50,6487.0,6,6,6
6,7,6251.0,7,7,7
7,15,6167.0,8,8,8
8,32,6155.0,9,9,9
9,47,6051.0,10,10,10


**Q9 explanation:**

*ROW_NUMBER() assigns a unique sequential number to every row, so tied revenues receive different row numbers.*

*RANK() assigns the same rank to tied revenues but skips the next rank(s) after the tie.*

*DENSE_RANK() assigns the same rank to tied revenues without skipping any subsequent ranks..*

### Q10 (15 marks) &mdash; Group total vs running total

For **driver_id 14**, list every completed ride in date order with four columns:

- `ride_date`, `fare`
- `driver_total` &mdash; grand total stamped on every row (`SUM(fare) OVER (PARTITION BY driver_id)`)
- `running_total` &mdash; cumulative up to this ride (`SUM(fare) OVER (PARTITION BY driver_id ORDER BY ride_date, ride_id)`)

Round both totals to 0 decimals. Then explain what makes the two columns behave differently.

In [13]:
# Q10
q("""
    SELECT
        ride_date,
        fare,
        ROUND(SUM(fare) OVER (PARTITION BY driver_id)) AS driver_total,
        ROUND(
            SUM(fare) OVER (
                PARTITION BY driver_id
                ORDER BY ride_date, ride_id
            )
        ) AS running_total
    FROM rides
    WHERE driver_id = 14
      AND status = 'completed'
    ORDER BY ride_date, ride_id
""")


,ride_date,fare,driver_total,running_total
0,2025-01-01,442.32,8494.0,442.0
1,2025-01-12,242.82,8494.0,685.0
2,2025-01-16,479.45,8494.0,1165.0
3,2025-01-18,474.29,8494.0,1639.0
4,2025-01-19,323.25,8494.0,1962.0
...,...,...,...,...
18,2025-03-26,554.31,8494.0,7066.0
19,2025-03-28,429.33,8494.0,7496.0
20,2025-03-29,269.95,8494.0,7766.0
21,2025-03-30,477.55,8494.0,8243.0


**Q10 explanation:**

*driver_total calculates the total fare from all completed rides for driver 14 and repeats the same value on every row. running_total calculates a cumulative total in ride order, so it increases with each completed ride until it reaches the driver's total.*

---
## Section D &mdash; Capstone (10 marks)

### Q11 (10 marks) &mdash; Top earner per city

In a **single query** (CTEs + multiple joins + window function): for each city, find the **top-earning driver** based on revenue from completed rides taken by riders based in that city.

Output: `city_name`, `driver` (name), `vehicle_type`, rounded `revenue`. Order by revenue descending.

> **Hint:** Joins needed: `rides` &rarr; `drivers` (driver name), `rides` &rarr; `riders` &rarr; `cities` (rider's city). Aggregate per `(city, driver)`, then `RANK() OVER (PARTITION BY city ...)` to find #1.

In [14]:
# Q11
q("""
    WITH city_driver_revenue AS (
        SELECT
            c.city_name,
            r.driver_id,
            d.name AS driver,
            d.vehicle_type,
            SUM(r.fare) AS revenue
        FROM rides AS r
        INNER JOIN drivers AS d
            ON r.driver_id = d.driver_id
        INNER JOIN riders AS rd
            ON r.rider_id = rd.rider_id
        INNER JOIN cities AS c
            ON rd.city_id = c.city_id
        WHERE r.status = 'completed'
        GROUP BY
            c.city_name,
            r.driver_id,
            d.name,
            d.vehicle_type
    ),
    ranked_drivers AS (
        SELECT
            city_name,
            driver,
            vehicle_type,
            ROUND(revenue) AS revenue,
            RANK() OVER (
                PARTITION BY city_name
                ORDER BY revenue DESC
            ) AS rank_
        FROM city_driver_revenue
    )
    SELECT
        city_name,
        driver,
        vehicle_type,
        revenue
    FROM ranked_drivers
    WHERE rank_ = 1
    ORDER BY revenue DESC
""")

,city_name,driver,vehicle_type,revenue
0,Izmir,Gizem Yilmaz,xl,2794.0
1,Antalya,Zeynep Polat,comfort,2687.0
2,Istanbul,Melis Yilmaz,xl,2436.0
3,Bursa,Gizem Yilmaz,xl,2219.0
4,Ankara,Merve Ozdemir,comfort,1767.0


---

<div style="background:#111A33;border-radius:10px;padding:14px 18px;color:#C7CEDB;font-size:13px">
<b style="color:#F5A623">Depthware</b> &nbsp;&middot;&nbsp; Modern Data Engineering &nbsp;&middot;&nbsp; Week 5 Homework &nbsp;&middot;&nbsp; 100 marks
</div>